In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv("engine_data.csv")

def section(title):
    n = 10
    print("\n" + "#" * n + f" {title} " + "#" * n + "\n")

In [ ]:
# Verteilung der Werte
print(df.describe().round(2))

# Überblick über den Datensatz:
section("INFO")
df.info()

In [ ]:
# Die ersten 5 Zeilen der CSV
section("Head")
print(df.head())

# Fehlende Werte sehen
section("isnull ?")
print(df.isnull().sum())
# Fehlende Werte ? 
if df.isnull().sum().sum() == False:
    print("\n" + "Es gibt keine fehlende Werte ")
else:
    print("\n" + "Es gibt fehlende Werte")
    
# Verteilung des Engine Zustands  
section("Engine Condition Verteilung")
print(df['Engine Condition'].value_counts())   
print(df['Engine Condition'].value_counts(normalize=True))

In [ ]:
# Korrelationsmatrix berechnen und anzeigen
# Damit erfahren wir, welche Spalten zusammenhängen.
# Beispiel Steigt die Öltemperatur immer dann, wenn auch die Kühlmitteltemperatur steigt?
corr = df.corr()
print(corr)

sns.heatmap(corr, annot=True, fmt=".2f")
plt.title("Korrelationsmatrix")
plt.show()

# Für uns ist besonders die letzte spalte wichtig (Engine Condition)
# Engine rpm: ca. -0.27 -> Je höher die Drehzahl, desto schlechter ist es für Engine Condition 1 -> 0 (nicht gesund).
# Fuel pressure: ca. +0.12 -> Etwas höherer Kraftstoffdruck bedeutet häufiger gesund.
# Lub oil pressure: ca. +0.06 -> Ein bisschen mehr Öldruck ist eher ein Zeichen für Gesundheit.
# Öltemperatur und Kühlmitteltemperatur: kleine negative Werte (-0.05 bis -0.09)
# Wenn die Temperaturen hoch sind ist der Motor einen Tick öfter ungesund.
# Meiner Meinung nach ergibt das keinen Sinn da die Motortemperatur sehr wichtig ist zu hohe Temperaturen können sehr schnell zu schweren Motorschäden führen.
# Werte sehr nah bei 0 (z.B. -0.02) bedeuten Kein klarer Zusammenhang fast egal für die Vorhersage.


In [ ]:
# X = alle Sensorwerte (rpm, Druck, Temperaturen usw.)
# y = Engine Condition (0 = ungesund, 1 = gesund)
x = df.drop(columns = ["Engine Condition"])
y = df["Engine Condition"]

# Daten aufteilen in Training und Test
# 85% der Daten sind zum trainieren -> daraus lernt unser Modell.
# 15% der Daten sind zum testen um zu gucken ob unser Modell was gelernt hat.
# stratify=y um das Verhältnis zwischen 0/1 gleich zu halten.
# random_state=42 immer gleiche Zufallswürfel, so bemerken wir, ob unser Code was gebracht hat

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.15, stratify=y, random_state=42)

# Erstellen unser Modell
rf = RandomForestClassifier(
    n_estimators=500,       # Unser Forest hat 500 Bäume, macht Code langsamer, aber robuster
    min_samples_leaf=2,
    random_state=42,     
    class_weight="balanced",)

# Modell trainieren, es lernt, wie die Sensorwerte mit Engine Condition 0/1 zusammenhängen                           
rf.fit(x_train, y_train)

# Jetzt mit Testdaten überprüfen ob unser Modell das Gelernte richtig anwenden kann 
y_predict = rf.predict(x_test)

# zeigt wie gut unser Modell ist indem wir die Lösungen mit den Vorhersagen vergleichen
section("Bericht")
print(classification_report(y_test, y_predict))
section("Konfisionsmatrix")
print(confusion_matrix(y_test, y_predict))
# Insgesamt liegt das Modell bei 65 % Genauigkeit
# Gesunde Motoren 1 werden gut erkannt
# Ungesunde Motoren 0 werden sehr schlecht erkannt
# Für eine erste Version ist das Ergebnis okay für die echte Praxis nicht geeignet.
